# H2O AutoML 라이브 데모

> RAPIDS LAB 세미나 — Pandas 이후, RAPIDS 이전의 자동화 병렬 ML 프레임워크

---

## 발표 흐름
1. **[Phase 1]** H2O 초기화 + 데이터 로드 + AutoML 학습 시작 (~1분)
2. **[Phase 2]** 학습 대기 중 → README / 슬라이드로 H2O AutoML 설명 (~5분)
3. **[Phase 3]** 학습 완료 후 → 결과 분석 (~4분)

---
## Phase 1: 런타임 시작
### 1-1. H2O 클러스터 초기화

In [1]:
import h2o
from h2o.automl import H2OAutoML

# H2O 클러스터 시작 (로컬 JVM)
# nthreads=-1: 모든 CPU 코어 사용 (멀티코어 병렬 처리의 핵심)
# max_mem_size: JVM 힙 메모리 (XGBoost는 별도 메모리 필요하므로 전체의 2/3 권장)
h2o.init(nthreads=-1, max_mem_size="4G")

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.18" 2026-01-20; OpenJDK Runtime Environment Homebrew (build 17.0.18+0); OpenJDK 64-Bit Server VM Homebrew (build 17.0.18+0, mixed mode, sharing)
  Starting server from /Users/harrisonkim/Documents/project/H2O_AutoML/.venv/lib/python3.9/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/n8/4qyw51dd5cz_tjkpxs_r5fjh0000gn/T/tmp7fybr2os
  JVM stdout: /var/folders/n8/4qyw51dd5cz_tjkpxs_r5fjh0000gn/T/tmp7fybr2os/h2o_harrisonkim_started_from_python.out
  JVM stderr: /var/folders/n8/4qyw51dd5cz_tjkpxs_r5fjh0000gn/T/tmp7fybr2os/h2o_harrisonkim_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,02 secs
H2O_cluster_timezone:,Asia/Seoul
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,8 days
H2O_cluster_name:,H2O_from_python_harrisonkim_wyn6mb
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,4 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


### 1-2. 데이터셋 로드

**Credit Card Transactions Dataset**
- 카드 거래 데이터 (정상/사기 라벨 포함)
- 타겟: `is_fraud` (사기 여부, 0/1)
- 약 129만 건 레코드
- 로컬 파일: `datasets/credit_card_transactions.csv`

In [ ]:
# 신용카드 거래 데이터셋 로드
# 로컬: datasets/credit_card_transactions.csv
credits = h2o.import_file("../datasets/credit_card_transactions.csv")

print(f"데이터 크기: {credits.shape}")
print(f"컬럼: {credits.columns}")
credits.head(5)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
데이터 크기: (1296675, 24)
컬럼: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'merch_zipcode']


Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,merch_zipcode
0,2019-01-01 00:00:18,2.70319e+15,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09 00:00:00,0b242abb623afc578575680df30655b9,1.32538e+09,36.0113,-82.0483,0,28705
1,2019-01-01 00:00:44,6.30423e+11,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.21,149,Special educational needs teacher,1978-06-21 00:00:00,1f76529f8574734946361c461b024d99,1.32538e+09,49.159,-118.186,0,nan
2,2019-01-01 00:00:51,3.88595e+13,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.262,4154,Nature conservation officer,1962-01-19 00:00:00,a1a22d70485983eac12b5b88dad1cf95,1.32538e+09,43.1507,-112.154,0,83236
3,2019-01-01 00:01:16,3.53409e+15,"fraud_Kutch, Hermiston and Farrell",gas_transport,45,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.114,1939,Patent attorney,1967-01-12 00:00:00,6b849c168bdad6f867558c3793159a81,1.32538e+09,47.0343,-112.561,0,nan
4,2019-01-01 00:03:06,3.75534e+14,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28 00:00:00,a41d7549acf90789359a9aa5346dcb46,1.32538e+09,38.675,-78.6325,0,22844


In [6]:
# 데이터 기본 탐색
credits.describe()

Rows:1296675
Cols:24

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,merch_zipcode
type,int,time,int,enum,enum,real,enum,enum,enum,enum,enum,enum,int,real,real,int,enum,time,string,int,real,real,int,int
mins,0.0,1546300818000.0,60416207185.0,,,1.0,,,,,,,1257.0,20.0271,-165.6723,23.0,,-1425513600000.0,NaN,1325376018.0,19.027785,-166.671242,0.0,1001.0
mean,648337.0,1570106848070.214,4.171920420797263e+17,,,70.35103545607045,,,,,,,48800.671097229395,38.53762161489967,-90.22633537864152,88824.44056297842,,118522975017.17857,NaN,1349243636.7261238,38.53733804469971,-90.2264647989728,0.005788651743883394,46825.754150532994
maxs,1296674.0,1592741617000.0,4.992346398065154e+18,,,28948.9,,,,,,,99783.0,66.6933,-67.9503,2906700.0,,1106956800000.0,NaN,1371816817.0,67.510267,-66.950902,1.0,99403.0
sigma,374317.9744882685,12855448282.253107,1.3088064470002404e+18,,,160.3160385715277,,,,,,,26893.222476485884,5.075808438803931,13.75907694648631,301956.360688751,,548805592934.60864,NaN,12841278.423358439,5.109788369679176,13.771090564792422,0.07586268973125163,25834.001159599862
zeros,1,0,0,,,0,,,,,,,0,0,0,0,,0,0,0,0,0,1289169,0
missing,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,195973
0,0.0,2019-01-01 00:00:18,2703186189652095.0,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654.0,36.0788,-81.1781,3495.0,"Psychologist, counselling",1988-03-09 00:00:00,0b242abb623afc578575680df30655b9,1325376018.0,36.011293,-82.048315,0.0,28705.0
1,1.0,2019-01-01 00:00:44,630423337322.0,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160.0,48.8878,-118.2105,149.0,Special educational needs teacher,1978-06-21 00:00:00,1f76529f8574734946361c461b024d99,1325376044.0,49.159046999999994,-118.186462,0.0,nan
2,2.0,2019-01-01 00:00:51,38859492057661.0,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252.0,42.1808,-112.262,4154.0,Nature conservation officer,1962-01-19 00:00:00,a1a22d70485983eac12b5b88dad1cf95,1325376051.0,43.150704,-112.154481,0.0,83236.0


### 1-3. AutoML 학습 시작

**핵심 포인트**: 이 셀 하나로 6가지 알고리즘 + 하이퍼파라미터 튜닝 + 앙상블이 전부 자동 수행됩니다.

In [ ]:
# 예측 변수 / 응답 변수 설정
y = "is_fraud"
x = credits.columns
x.remove(y)

# 누수/식별자 성격 컬럼 제외 (권장)
drop_cols = ["Unnamed: 0", "trans_num"]
x = [c for c in x if c not in drop_cols]

# 응답 변수를 factor로 변환 (이진분류)
credits[y] = credits[y].asfactor()

# Train/Test 분리 (80/20)
train, test = credits.split_frame(ratios=[0.8], seed=42)
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"y: {y}, x 개수: {len(x)}")

In [ ]:
%%time

# ============================================================
# H2O AutoML 실행 — 이것이 전부입니다
# ============================================================
# max_runtime_secs=300 : 5분 동안 가능한 많은 모델 탐색
# seed=42             : 재현성 (단, DeepLearning은 비결정적)
# nfolds=5            : 5-fold cross-validation (기본값)
#
# 내부에서 수행되는 작업:
#   1) XGBoost 사전 정의 모델 3개
#   2) GLM (lambda_search 포함)
#   3) Random Forest (DRF)
#   4) GBM 5개
#   5) Deep Neural Network
#   6) Extremely Randomized Trees (XRT)
#   7) XGBoost/GBM/DNN 랜덤 그리드 서치
#   8) Stacked Ensemble 2종 (Best of Family + All Models)
# ============================================================

aml = H2OAutoML(
    max_runtime_secs=300,
    seed=42,
    project_name="credits_automl",
    sort_metric="AUCPR",
    balance_classes=True,
    verbosity="info",
)

aml.train(x=x, y=y, training_frame=train)

print("\n=== AutoML 학습 완료 ===")

---

> **여기서 학습이 돌아가는 동안 README.md로 이동하여 H2O AutoML 특징을 설명합니다.**
> 
> (Phase 2: 약 5분 설명)

---

## Phase 3: 결과 분석

### 3-1. 리더보드 확인

AutoML이 학습한 모든 모델의 성능 순위표입니다.

In [ ]:
# 리더보드 — 모든 모델의 CV 성능 비교
lb = aml.leaderboard
print(f"총 {lb.nrows}개 모델 학습 완료\n")
lb.head(rows=lb.nrows)

In [ ]:
# 확장 리더보드: 학습 시간 + 예측 속도 포함
lb_ext = h2o.automl.get_leaderboard(aml, extra_columns="ALL")
lb_ext.head(rows=lb_ext.nrows)

### 3-2. 최고 모델 (Leader) 분석

In [ ]:
# AutoML이 선택한 최고 모델
leader = aml.leader
print(f"Best Model: {leader.model_id}")
print(f"Algorithm:  {leader.algo}")
print(f"\n--- Model Parameters ---")
leader.params

In [ ]:
# 테스트 데이터에서 성능 평가
perf = leader.model_performance(test)
print(perf)

In [ ]:
# AUC 확인
print(f"Test AUC: {perf.auc():.4f}")
print(f"Test Logloss: {perf.logloss():.4f}")

### 3-3. 알고리즘별 최고 모델 비교

In [ ]:
# 알고리즘별 최고 모델 가져오기
algos = ["xgboost", "gbm", "glm", "deeplearning", "drf", "stackedensemble"]

print(f"{'Algorithm':<20} {'Model ID':<55} {'AUC':>8}")
print("-" * 85)

for algo in algos:
    try:
        m = aml.get_best_model(algorithm=algo)
        p = m.model_performance(test)
        print(f"{algo:<20} {m.model_id:<55} {p.auc():>8.4f}")
    except Exception as e:
        print(f"{algo:<20} {'N/A':<55} {str(e)[:8]:>8}")

### 3-4. Feature Importance

In [ ]:
# 리더 모델의 변수 중요도 (tree 기반 모델인 경우)
# StackedEnsemble인 경우 base learner 중 하나를 사용
best_non_ensemble = aml.get_best_model(algorithm="gbm")
if best_non_ensemble is not None:
    best_non_ensemble.varimp_plot()

### 3-5. 예측 수행

In [ ]:
# 테스트 데이터 예측
preds = aml.predict(test)
preds.head(10)

### 3-6. AutoML 이벤트 로그

학습 과정에서 어떤 일이 일어났는지 확인합니다.

In [ ]:
# AutoML 이벤트 로그
aml.event_log.head(rows=30)

In [ ]:
# 학습 정보 요약
print("=== Training Info ===")
for k, v in aml.training_info.items():
    print(f"  {k}: {v}")

---

## 정리

| 항목 | 결과 |
|------|------|
| 코드 라인 수 | AutoML 실행: **단 6줄** |
| 탐색 알고리즘 | GBM, XGBoost, DRF, GLM, DeepLearning, StackedEnsemble |
| 자동 수행 항목 | 하이퍼파라미터 튜닝, Cross-validation, Early Stopping, 앙상블 |
| 병렬 처리 | JVM 기반 멀티코어 자동 활용 (설정 불필요) |

> **"5분, 6줄의 코드로 수십 개의 모델을 자동 학습하고 최적 앙상블까지 구성"**
> — 이것이 RAPIDS 이전 시대, H2O AutoML이 제시한 답입니다.

In [ ]:
# 클러스터 종료
# h2o.cluster().shutdown()